# 02 — Disponibilité Sentinel-2

Diagnostic de disponibilité du catalogue Sentinel-2 L2A sur l'AOI SeineCrops
(Pays de Caux + plateau de Neubourg, ~3 349 km²) pour la fenêtre temporelle
septembre N → décembre N+1 (~16 mois), alignée sur le millésime RPG N+1.

Ce notebook constitue un **diagnostic catalogue pur** : aucune image n'est
téléchargée. Les métadonnées sont interrogées via l'API OData du Copernicus
Data Space Ecosystem (CDSE) pour les quatre tuiles Sentinel-2 retenues.

---

## AOI et tuiles

L'AOI couvre deux entités géographiques séparées par la Seine :

| Paire | Tuiles | Zone couverte |
|-------|--------|---------------|
| Nord | 30UYA, 31UCR | Pays de Caux (rive droite) |
| Oest   | 30UYV, 31UCQ | Plateau de Neubourg (rive gauche) |

Chaque tuile couvre environ 50 % de l'AOI. La couverture **complète** de l'AOI
pour un jour donné requiert au moins une scène exploitable dans chacune des deux
paires.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| S2.1 | Authentification CDSE (OAuth, gestion du rafraîchissement de token) |
| S2.2 | Requête catalogue OData — boucle sur les 4 tuiles, pagination |
| S2.3 | Structuration en DataFrame — colonnes `pair`, `date`, `cloud_cover_catalogue`, `f_valid_aoi` |
| S2.4 | Statistiques mensuelles — couverture partielle et couverture AOI complète |
| S2.5 | Visualisation et rapport — histogramme, `AVAILABILITY_REPORT.json` |

---

## Niveau 2 (sprint S2)

La colonne `f_valid_aoi` est provisionnée ici mais restera `NaN` jusqu'au
sprint S2, où la bande SCL de chaque scène sera téléchargée pour calculer
la fraction de pixels valides **sur l'AOI** (classes SCL invalides : 3, 8, 9,
10, 11). Ce calcul remplace le `cloud_cover_catalogue` (métrique tuile entière,
aveugle à la localisation des nuages) par une disponibilité réelle ∈ [0, 1].

---

## Références

- [Documentation OData CDSE](https://documentation.dataspace.copernicus.eu/APIs/OData.html)
- [Sentinel-2 L2A — bande SCL (classes et usage)](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html)
- [HR-VPP — manuel produit trajectoires saisonnières et paramètres VPP (PDF)](https://land.copernicus.eu/en/technical-library/product-user-manual-of-seasonal-trajectories/@@download/file)
- `01_ingestion_rpg.ipynb` — sprint S1, ingestion RPG en PostGIS

L'histogramme représente pour chaque mois trois métriques complémentaires :
le nombre de scènes disponibles
(`n_scenes`), le nombre de jours avec au moins
une scène (`days_covered`, couverture partielle) et le nombre de jours où les
deux paires nord et sud sont simultanément couvertes (`days_full_aoi`,
couverture quasi complète). Aucun filtre de couverture nuageuse — disponibilité
catalogue brute.

En clôture de section, `df_dedup` est persisté dans
`data/raw/s2/catalogue_dedup.parquet` (non versionné). Ce fichier est la
dépendance directe de `03_series_s2.ipynb` — il porte les identifiants CDSE
de chaque scène nécessaires au téléchargement des bandes spectrales.

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

import os
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

# Résolution du .env via le marqueur .projectroot (cohérent avec S1)
def _find_project_root() -> Path:
    marker = ".projectroot"
    p = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    return p

PROJECT_ROOT = _find_project_root()

### S2.1 — Authentification CDSE

Obtention et rafraîchissement d'un token OAuth auprès du Copernicus Data Space
Ecosystem. Le token est requis pour les téléchargements (bande SCL en sprint S2) ;
il est déjà mis en place ici pour ne pas avoir à le rajouter en cours de pipeline.

L'endpoint d'authentification est commun à toutes les APIs CDSE (OData,
téléchargement, S3). Les credentials sont lus depuis `.env` via `python-dotenv`,
cohérent avec la pratique établie en S1.

**Variables `.env` attendues**

| Variable | Description |
|----------|-------------|
| `CDSE_USER` | Adresse e-mail du compte CDSE |
| `CDSE_PASSWORD` | Mot de passe du compte CDSE |

**Comportement du token**

Le token a une durée de vie d'environ 10 minutes (`expires_in` retourné par
l'endpoint). La fonction `refresh_cdse_token` est appelée avant chaque page de
résultats en S2.2 pour éviter une expiration silencieuse en cours de boucle.

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

# Vérification immédiate avant d'aller plus loin
missing = [v for v in ("CDSE_USER", "CDSE_PASSWORD") if not os.getenv(v)]
if missing:
    raise EnvironmentError(
        f"Variables manquantes dans {PROJECT_ROOT / '.env'} : {missing}\n"
        f"Répertoire courant : {Path.cwd()}\n"
        f"Racine projet détectée : {PROJECT_ROOT}"
    )

CDSE_TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu"
    "/auth/realms/CDSE/protocol/openid-connect/token"
)

def get_cdse_token() -> dict:
    """Retourne un dict {access_token, refresh_token, expires_at}."""
    resp = requests.post(
        CDSE_TOKEN_URL,
        data={
            "grant_type": "password",
            "client_id":  "cdse-public",
            "username":   os.environ["CDSE_USER"],
            "password":   os.environ["CDSE_PASSWORD"],
        },
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()
    return {
        "access_token":  payload["access_token"],
        "refresh_token": payload["refresh_token"],
        "expires_at":    time.time() + payload["expires_in"] - 30,  # marge 30 s
    }

def refresh_cdse_token(token_dict: dict) -> dict:
    """Rafraîchit le token si expiré, sinon le retourne tel quel."""
    if time.time() < token_dict["expires_at"]:
        return token_dict
    resp = requests.post(
        CDSE_TOKEN_URL,
        data={
            "grant_type":    "refresh_token",
            "client_id":     "cdse-public",
            "refresh_token": token_dict["refresh_token"],
        },
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()
    return {
        "access_token":  payload["access_token"],
        "refresh_token": payload["refresh_token"],
        "expires_at":    time.time() + payload["expires_in"] - 30,
    }

token = get_cdse_token()
print(f"Token obtenu, expire dans ~{int(token['expires_at'] - time.time())} s")

### S2.2 — Requête catalogue multi-tuiles

Interrogation du catalogue OData CDSE pour les 4 tuiles couvrant l'AOI,
sur la fenêtre temporelle complète septembre N → décembre N+1.
Aucun filtre de couverture nuageuse n'est appliqué : toutes les scènes L2A
disponibles sont récupérées. Le `cloud_cover_catalogue` est conservé à titre
informatif ; le filtrage réel sera effectué en sprint S2 sur la bande SCL.

La pagination est gérée automatiquement (`$top` / `$skip`). Le token est
rafraîchi avant chaque page pour éviter une expiration silencieuse en cours
de boucle (~10 min de durée de vie).

La colonne `pair` encode l'appartenance géographique de chaque tuile :

| Tuile | Paire | Zone |
|-------|-------|------|
| 30UYA | nord | Pays de Caux |
| 31UCR | nord | Pays de Caux |
| 30UYV | sud | Plateau de Neubourg |
| 31UCQ | sud | Plateau de Neubourg |

In [ ]:
# ── Paramètres de campagne ───────────────────────────────────────────
from datetime import date

ANNEE_REFERENCE = 2024          # millésime RPG N+1

DATE_START = f"{ANNEE_REFERENCE - 1}-09-01T00:00:00.000Z"
DATE_END   = f"{ANNEE_REFERENCE}-12-31T23:59:59.999Z"

TUILES = {
    "30UYA": "nord",
    "31UCR": "nord",
    "30UYV": "sud",
    "31UCQ": "sud",
}

print(f"Fenêtre temporelle : {DATE_START[:10]} → {DATE_END[:10]}")
print(f"Tuiles : {list(TUILES.keys())}")

In [ ]:
# ── Requête catalogue OData, une tuile ───────────────────────────────
ODATA_BASE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

def query_s2_catalogue(
    tile_id: str,
    date_start: str,
    date_end: str,
    token_dict: dict,
    page_size: int = 1000,
) -> list[dict]:
    """
    Interroge le catalogue OData pour une tuile S2 L2A.
    Pagine automatiquement jusqu'à épuisement des résultats.
    Aucun filtre cloudCover : toutes les scènes disponibles sont retournées.
    """
    filter_str = (
        f"Collection/Name eq 'SENTINEL-2' and "
        f"Attributes/OData.CSC.StringAttribute/any("
        f"att:att/Name eq 'productType' and "
        f"att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
        f"Attributes/OData.CSC.StringAttribute/any("
        f"att:att/Name eq 'tileId' and "
        f"att/OData.CSC.StringAttribute/Value eq '{tile_id}') and "
        f"ContentDate/Start ge {date_start} and "
        f"ContentDate/Start le {date_end}"
    )

    results, skip = [], 0
    while True:
        token_dict = refresh_cdse_token(token_dict)
        resp = requests.get(
            ODATA_BASE,
            params={
                "$filter":  filter_str,
                "$orderby": "ContentDate/Start asc",
                "$top":     page_size,
                "$skip":    skip,
                "$expand":  "Attributes",
            },
            headers={"Authorization": f"Bearer {token_dict['access_token']}"},
            timeout=60,
        )
        resp.raise_for_status()
        batch = resp.json().get("value", [])
        results.extend(batch)
        if len(batch) < page_size:
            break
        skip += page_size

    return results

In [ ]:
# ── Boucle catalogue ─────────────────────────────────────────────────
import time

raw_results = {}   # {tile_id: [dict, ...]}

for tile_id, pair in TUILES.items():
    t0 = time.time()
    scenes = query_s2_catalogue(
        tile_id=tile_id,
        date_start=DATE_START,
        date_end=DATE_END,
        token_dict=token,
    )
    raw_results[tile_id] = scenes
    print(f"{tile_id} ({pair:4s}) → {len(scenes):3d} scènes  [{time.time()-t0:.1f}s]")

total = sum(len(v) for v in raw_results.values())
print(f"\nTotal brut (toutes tuiles, doublons inclus) : {total} scènes")

### S2.3 — Structuration en DataFrame

Mise en forme des résultats bruts OData en DataFrame pandas.
Chaque ligne correspond à une scène (une tuile, une date).

Les attributs `tileId`, `cloudCover` et `orbitNumber` sont extraits
du champ `Attributes` (liste de dicts retournée par `$expand=Attributes`).
La colonne `f_valid_aoi` est provisionnée à `NaN` — elle sera calculée
en sprint S2 à partir de la bande SCL sur l'AOI.

In [ ]:
# ── Structuration en DataFrame ────────────────────────────────────────
import pandas as pd
import numpy as np

def extract_attribute(attributes: list[dict], name: str):
    """Extrait la valeur d'un attribut OData par son nom."""
    for attr in attributes:
        if attr.get("Name") == name:
            return attr.get("Value")
    return None

records = []
for tile_id, pair in TUILES.items():
    for scene in raw_results[tile_id]:
        attrs = scene.get("Attributes", [])
        records.append({
            "product_id":            scene["Id"],
            "scene_id":              scene["Name"],
            "tile_id":               tile_id,
            "pair":                  pair,
            "datetime_utc":          pd.to_datetime(scene["ContentDate"]["Start"]),
            "cloud_cover_catalogue": extract_attribute(attrs, "cloudCover"),
            "orbit_relative":        extract_attribute(attrs, "relativeOrbitNumber"),
            "f_valid_aoi":           np.nan,
        })

df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["datetime_utc"].dt.date)
df = df.sort_values(["datetime_utc", "tile_id"]).reset_index(drop=True)

print(f"Scènes totales : {len(df)}")
print(f"Fenêtre : {df['date'].min()} → {df['date'].max()}")
print(f"\nScènes par tuile :")
print(df.groupby(["tile_id", "pair", "orbit_relative"])["scene_id"].count()
        .rename("n_scènes").to_string())

In [ ]:
# ── Contrôle qualité du DataFrame ────────────────────────────────────
print(df.dtypes)
print(f"\nValeurs nulles :\n{df.isnull().sum()}")
print(f"\nAperçu :")
df.head(8)

### S2.4 — Statistiques mensuelles

Calcul de la disponibilité par mois sur deux métriques complémentaires :

- **Couverture partielle** : au moins une scène disponible ce jour-là
  sur l'une quelconque des 4 tuiles
- **Couverture quasi complète** : au moins une scène disponible ce jour-là
  dans chacune des deux paires — (30UYA **ou** 31UCR) **et**
  (30UYV **ou** 31UCQ) — condition nécessaire pour couvrir l'AOI entière (30UYA) ou l'AOI hors pointe de Caux (31UCR)

In [ ]:
# Chercher les doublons tuile + date
dupes = (
    df.groupby(["tile_id", "date"])["scene_id"]
    .count()
    .rename("n")
    .loc[lambda x: x > 1]
)
print(f"Combinaisons tuile×date avec plus d'une scène : {len(dupes)}")
print(dupes.sort_values(ascending=False).head(20))

In [ ]:
# ── Déduplication : baseline le plus récent par tuile × date ─────────
# Le baseline est encodé dans le nom : S2X_MSIL2A_YYYYMMDD..._N0510_...
# La date de génération du produit (dernier segment) est aussi un critère
# On trie par scene_id décroissant — le baseline N0511 > N0510 > ...
# et la date de génération la plus récente arrive en dernier alphabétiquement

df_dedup = (
    df.sort_values("scene_id", ascending=False)
    .drop_duplicates(subset=["tile_id", "date"], keep="first")
    .sort_values(["datetime_utc", "tile_id"])
    .reset_index(drop=True)
)

print(f"Scènes avant déduplication : {len(df)}")
print(f"Scènes après déduplication : {len(df_dedup)}")
print(f"Doublons supprimés         : {len(df) - len(df_dedup)}")

# Vérification
remaining = (
    df_dedup.groupby(["tile_id", "date"])["scene_id"]
    .count()
    .loc[lambda x: x > 1]
)
print(f"Combinaisons tuile×date restantes avec n>1 : {len(remaining)}")

In [ ]:
# ── Disponibilité journalière ─────────────────────────────────────────
# Couverture partielle : au moins une scène sur l'une quelconque des 4 tuiles
daily = (
    df_dedup.groupby("date")["scene_id"]
    .count()
    .rename("n_scenes")
)

# Couverture quasi complète : au moins une scène dans chaque paire (nord AND sud)
# Un jour est quasi complet si la paire nord ET la paire sud sont toutes deux
# représentées — condition nécessaire pour couvrir l'AOI entière.
pairs_per_day = (
    df_dedup.groupby("date")["pair"]
    .apply(lambda x: set(x.unique()))
)
daily_full_aoi = pairs_per_day.apply(
    lambda pairs: {"nord", "sud"}.issubset(pairs)
).rename("full_aoi")

daily_covered      = int((daily > 0).sum())
daily_full_covered = int(daily_full_aoi.sum())
print(f"Jours avec au moins une scène (couverture partielle)     : {daily_covered}")
print(f"Jours avec nord ET sud couverts (couverture quasi compl.): {daily_full_covered}")
print(f"Jours totaux dans la fenêtre                             : {len(daily)}")

In [ ]:
# ── Agrégation mensuelle ──────────────────────────────────────────────
# Joindre la série daily_full_aoi pour l'agréger au même pas mensuel
daily_combined = pd.concat([daily, daily_full_aoi], axis=1).fillna({"full_aoi": False})

monthly = (
    daily_combined
    .resample("MS")
    .agg(
        n_scenes=("n_scenes", "sum"),
        days_covered=("n_scenes", lambda x: (x > 0).sum()),
        days_full_aoi=("full_aoi", "sum"),
    )
    .assign(
        days_in_month=lambda x: x.index.days_in_month,
        pct_covered=lambda x: (x["days_covered"] / x["days_in_month"] * 100).round(1),
        pct_full_aoi=lambda x: (x["days_full_aoi"] / x["days_in_month"] * 100).round(1),
    )
)

print(monthly[
    ["n_scenes", "days_covered", "days_full_aoi", "days_in_month",
     "pct_covered", "pct_full_aoi"]
].to_string())

### S2.5 — Visualisation et rapport

Histogramme de disponibilité mensuelle et export du rapport de clôture
`AVAILABILITY_REPORT.json`, sur le modèle de `INGESTION_REPORT.json` (sprint S1).

L'histogramme représente pour chaque mois trois métriques complémentaires :
le nombre de scènes disponibles (`n_scenes`), le nombre de jours avec au moins
une scène (`days_covered`, couverture partielle) et le nombre de jours où les
deux paires nord et sud sont simultanément couvertes (`days_full_aoi`,
couverture quasi complète). Aucun filtre de couverture nuageuse — disponibilité
catalogue brute.

In [ ]:
# ── Histogramme de disponibilité ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax1 = plt.subplots(figsize=(14, 5))

x     = range(len(monthly))
width = 0.28
months_labels = [d.strftime("%b\n%Y") for d in monthly.index]

ax1.bar(
    [i - width for i in x], monthly["n_scenes"],
    width=width, color="#4C8BB5", label="Scènes catalogue (4 tuiles)"
)
ax1.bar(
    [i for i in x], monthly["days_covered"],
    width=width, color="#E07B39", label="Jours couverts (partiel)"
)
ax1.bar(
    [i + width for i in x], monthly["days_full_aoi"],
    width=width, color="#4CAF50", label="Jours AOI quasi complète (nord+sud)"
)

ax1.set_xticks(list(x))
ax1.set_xticklabels(months_labels, fontsize=8)
ax1.set_ylabel("Nombre")
ax1.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax1.set_ylim(0, monthly["n_scenes"].max() * 1.20)
ax1.legend(loc="upper left", fontsize=8)
ax1.set_title(
    f"Disponibilité Sentinel-2 L2A — AOI SeineCrops\n"
    f"Tuiles : 30UYA · 31UCR · 30UYV · 31UCQ  ·  "
    f"Fenêtre : {DATE_START[:10]} → {DATE_END[:10]}  ·  "
    f"Sans filtre nuage",
    fontsize=10,
)

# Annotation pct_full_aoi au-dessus de la barre verte
for i, (_, row) in enumerate(monthly.iterrows()):
    if row["days_full_aoi"] > 0:
        ax1.text(
            i + width, row["days_full_aoi"] + 0.4,
            f"{row['pct_full_aoi']:.0f}%",
            ha="center", va="bottom", fontsize=7, color="#2E7D32"
        )

ax1.axvline(x=3.5, color="grey", linewidth=0.8, linestyle="--")
ax1.text(3.6, ax1.get_ylim()[1] * 0.97, "→ 2024", fontsize=8, color="grey")

fig.tight_layout()

out_fig = PROJECT_ROOT / "data" / "raw" / "s2" / "availability_s2.png"
out_fig.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Histogramme sauvegardé : {out_fig}")

In [ ]:
# ── AVAILABILITY_REPORT.json ──────────────────────────────────────────
import json
from datetime import datetime, timezone
from decimal import Decimal

class DecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        return super().default(obj)

# Calendrier complet de la fenêtre
date_min = pd.Timestamp(DATE_START[:10])
date_max = pd.Timestamp(DATE_END[:10])
jours_fenetre = (date_max - date_min).days + 1

# Réindexer daily sur le calendrier complet (inclut les jours sans scène)
daily_full = daily.reindex(
    pd.date_range(date_min, date_max, freq="D"),
    fill_value=0
)
daily_full_aoi_reindexed = daily_full_aoi.reindex(
    pd.date_range(date_min, date_max, freq="D"),
    fill_value=False
)

jours_couverts = int((daily_full > 0).sum())
jours_full_aoi = int(daily_full_aoi_reindexed.sum())
pct_couverts   = round(jours_couverts / jours_fenetre * 100, 1)
pct_full_aoi   = round(jours_full_aoi / jours_fenetre * 100, 1)

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "annee_reference": ANNEE_REFERENCE,
    "fenetre": {"start": DATE_START, "end": DATE_END},
    "tuiles": {
        tile_id: {"pair": pair}
        for tile_id, pair in TUILES.items()
    },
    "orbites_relatives": (
        df_dedup.groupby("tile_id")["orbit_relative"]
        .apply(lambda x: sorted(x.unique().tolist()))
        .to_dict()
    ),
    "catalogue": {
        "scenes_brutes":      len(df),
        "scenes_dedup":       len(df_dedup),
        "doublons_supprimes": len(df) - len(df_dedup),
        "filtre_nuage":       None,
    },
    "disponibilite": {
        "jours_fenetre":   jours_fenetre,
        "jours_couverts":  jours_couverts,
        "pct_couverts":    pct_couverts,
        "jours_full_aoi":  jours_full_aoi,
        "pct_full_aoi":    pct_full_aoi,
        "note_full_aoi":   "Jours où les paires nord (30UYA|31UCR) ET sud (30UYV|31UCQ) sont simultanément couvertes",
        "mensuel": [
            {
                "mois":          row.Index.strftime("%Y-%m"),
                "n_scenes":      int(row.n_scenes),
                "days_covered":  int(row.days_covered),
                "days_full_aoi": int(row.days_full_aoi),
                "days_in_month": int(row.days_in_month),
                "pct_covered":   float(row.pct_covered),
                "pct_full_aoi":  float(row.pct_full_aoi),
            }
            for row in monthly.itertuples()
        ],
    },
    "niveau_2": {
        "statut": "non calculé",
        "description": (
            "f_valid_aoi sera calculé en sprint S2 par téléchargement "
            "de la bande SCL (60 m) et calcul de la fraction de pixels "
            "valides sur l'AOI (classes SCL invalides : 3, 8, 9, 10, 11)."
        ),
    },
}

out_json = PROJECT_ROOT / "data" / "raw" / "s2" / "AVAILABILITY_REPORT.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, cls=DecimalEncoder)

print(f"Rapport sauvegardé : {out_json}")
print(f"Jours couverts (partiel)   : {jours_couverts} / {jours_fenetre} ({pct_couverts} %)")
print(f"Jours AOI quasi complète   : {jours_full_aoi} / {jours_fenetre} ({pct_full_aoi} %)")

In [ ]:
# ── Persistence du catalogue dédupliqué (livrable de clôture nb02) ───
# df_dedup est la dépendance directe de 03_series_s2.ipynb.
# Il est persisté en Parquet plutôt qu'en CSV pour préserver les types
# (datetime64, float64) sans conversion et pour la performance en lecture.

out_parquet = PROJECT_ROOT / "data" / "raw" / "s2" / "catalogue_dedup.parquet"
df_dedup.to_parquet(out_parquet, index=False)

print(f"Catalogue dédupliqué sauvegardé : {out_parquet}")
print(f"Scènes : {len(df_dedup)} · Colonnes : {list(df_dedup.columns)}")